# HPO Dual-View gated (v2) - parameters transferred, not searched

The gated variant differs from the average-fusion variant only in the fusion
mechanism (a learned per-token gate instead of a fixed 50/50 average). Its
hyperparameters are **transferred verbatim** from `dual_view_v2` so that the
gate is the only thing that changes. No search runs here.

**Output:** `hpo/dual_view_gated_v2/best_params.json`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, json
sys.path.insert(0, "/content/drive/MyDrive/google_colab/kusa/v2_heldout")
from config import *
import utils_split as u

In [ ]:
# The average-fusion variant is the only one that is searched. Its result is
# read here and reused, so all three architectures share one training setup.
DV_PATH = best_params_path("dual_view_v2")
assert os.path.exists(DV_PATH), (
    "hpo/dual_view_v2/best_params.json is missing - run hpo_kusa_dual_view_v2 first.")
with open(DV_PATH, encoding="utf-8") as f:
    dv = json.load(f)

print("dual_view_v2 best params (source of truth):")
for k, v in dv.items():
    if not k.startswith("_"):
        print(f"  {k:14s} = {v}")
print(f"  (best HPO-slice value: {dv['_best_value']:.4f}, {dv['_n_trials']} trials)")

In [ ]:
VARIANT = "dual_view_gated_v2"

# Gated CV reads: batch_size, encoder_lr, head_lr, dropout, weight_decay,
# warmup_ratio, epochs. All copied straight from the average-fusion variant.
keys = ["batch_size", "encoder_lr", "head_lr", "dropout",
        "weight_decay", "warmup_ratio", "epochs"]
best = {k: dv[k] for k in keys}
best.update({
    "_variant":          VARIANT,
    "_transferred_from": "dual_view_v2",
    "_note":             "identical training setup; only the fusion gate differs",
    "_best_value":       dv["_best_value"],
    "_n_trials":         0,
})

out = best_params_path(VARIANT)
with open(out, "w", encoding="utf-8") as f:
    json.dump(best, f, indent=2, ensure_ascii=False)
print(json.dumps(best, indent=2, ensure_ascii=False))
print("\nsaved:", out)

assert_test_untouched(globals())